<a href="https://colab.research.google.com/github/shizoda/education/blob/main/machine_learning/Perceptron.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 単純パーセプトロンと重みの調整による学習

アヤメ（Iris）分類を行います。がく片の長さを $x_1$、がく片の幅を $x_2$ として、それらの値から品種 Setosa（クラス0）と品種 Versicolor（クラス1）のいずれであるかを判別します。データは非常に単純化されています。

同じ題材は https://shizoda.github.io/mlearn/ngram.html でも扱っており、こちらは 3 クラスです。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import warnings

warnings.filterwarnings("ignore")

# ネットワーク構造を描画する関数
def draw_perceptron_network():
    G = nx.DiGraph()

    nodes = [
        "x1\n(Sepal Length)",
        "x2\n(Sepal Width)",
        "1\n(Bias)",
        "y\n(Output)"
    ]
    for node in nodes:
        G.add_node(node)

    pos = {
        "x1\n(Sepal Length)": (0, 1.5),
        "x2\n(Sepal Width)": (0, 0.0),
        "1\n(Bias)": (0, -1.5),
        "y\n(Output)": (2, 0.0)
    }

    edges = [
        ("x1\n(Sepal Length)", "y\n(Output)"),
        ("x2\n(Sepal Width)", "y\n(Output)"),
        ("1\n(Bias)", "y\n(Output)")
    ]
    G.add_edges_from(edges)

    edge_labels = {
        ("x1\n(Sepal Length)", "y\n(Output)"): "w1",
        ("x2\n(Sepal Width)", "y\n(Output)"): "w2",
        ("1\n(Bias)", "y\n(Output)"): "b"
    }

    plt.figure(figsize=(7, 4))

    nx.draw_networkx_nodes(
        G, pos,
        node_color=["lightblue", "lightblue", "lightgray", "salmon"],
        node_size=3200,
        edgecolors="k"
    )

    nx.draw_networkx_edges(G, pos, arrowstyle="->", arrowsize=20, width=1.5)
    nx.draw_networkx_labels(G, pos, font_size=9)
    nx.draw_networkx_edge_labels(
        G, pos, edge_labels=edge_labels, font_size=14, label_pos=0.4
    )

    plt.axis("off")
    plt.show()

draw_perceptron_network()

## データを見る

アヤメの測定値である がく片の長さ（Sepal Length: $x_1$）と がく片の幅（Sepal Width: $x_2$）を標準化したデータセットを用いて、各データ点の分布を可視化します。

青色の点は品種 Setosa（クラス0）、赤色の点は品種 Versicolor（クラス1）を表します。データは上下や左右に単純に二分されておらず、両者を分離するためには斜めに横切る直線が必要となる配置になっています。

In [ ]:
# アヤメの実測データ（Sepal Length: x1 [cm], Sepal Width: x2 [cm]）
X = np.array([
    [5.1, 3.5],
    [4.9, 3.0],
    [4.7, 3.2],
    [4.6, 3.1],
    [5.0, 3.6],
    [7.0, 3.2],
    [6.4, 3.2],
    [6.9, 3.1],
    [5.5, 2.3],
    [6.5, 2.8]
])
y = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])  # 0: Setosa, 1: Versicolor

# 散布図の描画
plt.figure(figsize=(6, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], color="blue", label="Setosa (Class 0)", s=60, edgecolors="k")
plt.scatter(X[y == 1, 0], X[y == 1, 1], color="red", label="Versicolor (Class 1)", s=60, edgecolors="k")
plt.xlim(4.0, 8.0)
plt.ylim(1.5, 4.5)
plt.xlabel("x1: Sepal Length [cm]")
plt.ylabel("x2: Sepal Width [cm]")
plt.title("Raw Iris Sepal Dataset Distribution")
plt.grid(True, linestyle=":")
plt.legend(loc="lower right")
plt.show()

## 重みと決定境界の関係

2つのクラスを分離する決定境界は、入力 $x_1$、$x_2$ と係数である重み $w_1$、$w_2$ およびバイアス $b$ によって以下の式で表されます。

$$w_1x_1+w_2x_2+b=0$$

初期状態では、$w_1=0.0$、$w_2=1.0$、$b=0.0$ となっており、決定境界が水平線として引かれます。この設定では水平方向の分離しか行えないため、複数のデータ点が誤って分類されます。

スライダーを操作して係数 $w_1$、$w_2$、$b$ を調節し、誤分類数（Misclassifications）が 0 になる組み合わせを探してください。境界線や計算が変化するのは、数式の係数を調節している結果にすぎないことをグラフ上部の算出式で確認できます。

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def plot_perceptron_raw(w1, w2, b):
    x_min, x_max = 4.0, 8.0
    y_min, y_max = 1.5, 4.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))

    z = w1 * xx + w2 * yy + b
    Z = np.where(z >= 0, 1, 0)

    preds = np.where((w1 * X[:, 0] + w2 * X[:, 1] + b) >= 0, 1, 0)
    errors = np.sum(preds != y)

    plt.figure(figsize=(7, 6))
    plt.contourf(xx, yy, Z, alpha=0.2, cmap="coolwarm")

    if abs(w2) > 1e-5:
        x_vals = np.array([x_min, x_max])
        y_vals = -(w1 * x_vals + b) / w2
        plt.plot(x_vals, y_vals, color="black", linestyle="--", label="Decision Boundary")
    elif abs(w1) > 1e-5:
        x_val = -b / w1
        plt.axvline(x=x_val, color="black", linestyle="--", label="Decision Boundary")

    plt.scatter(X[y == 0, 0], X[y == 0, 1], color="blue", label="Setosa (Class 0)", s=60, edgecolors="k")
    plt.scatter(X[y == 1, 0], X[y == 1, 1], color="red", label="Versicolor (Class 1)", s=60, edgecolors="k")

    plt.xlim(x_min, x_max)
    plt.ylim(y_min, y_max)
    plt.xlabel("x1: Sepal Length [cm]")
    plt.ylabel("x2: Sepal Width [cm]")

    formula_str = f"{w1:.2f} * x1 + ({w2:.2f}) * x2 + ({b:.2f}) = 0"
    plt.title(f"Formula: {formula_str}\nMisclassifications: {errors} / {len(y)}", fontsize=11)
    plt.grid(True, linestyle=":")
    plt.legend(loc="lower right")
    plt.show()

# 初期値の設定（中心値からずらした値）
initial_w1 = -1.5
initial_w2 = 4.0
initial_b = -3.0

# スライダーの定義（中心値 ± 0.5 の範囲に制限）
w1_slider = widgets.FloatSlider(value=initial_w1, min=-2.2, max=-1.2, step=0.1, description="w1")
w2_slider = widgets.FloatSlider(value=initial_w2, min=4.5, max=5.5, step=0.1, description="w2")
b_slider = widgets.FloatSlider(value=initial_b, min=-3.5, max=-2.5, step=0.1, description="b")
reset_button = widgets.Button(description="Reset Parameters")

# リセット処理
def reset_slider_values(button):
    w1_slider.value = initial_w1
    w2_slider.value = initial_w2
    b_slider.value = initial_b

reset_button.on_click(reset_slider_values)

# UI要素の配置と可視化の実行
ui = widgets.VBox([w1_slider, w2_slider, b_slider, reset_button])
out = widgets.interactive_output(plot_perceptron_raw, {"w1": w1_slider, "w2": w2_slider, "b": b_slider})
w1, w2, w3 = w1_slider.value, w2_slider.value, b_slider.value

display(ui, out)

## 新規データの判別と所属確率の計算

未知のアヤメの個体の がく片の長さ $x_1$ と がく片の幅 $x_2$ を入力し、係数を指定して、そのデータが品種 Versicolor（クラス1）に属する確率を計算します。

確率はシグモイド関数を用いて以下の式で計算されます。

$$P(y=1) = \frac{1}{1 + e^{-(w_1x_1+w_2x_2+b)}}$$

算出された確率が 0.5 以上であれば品種 Versicolor（クラス1）、0.5 未満であれば品種 Setosa（クラス0）と分類されます。新たに入力したデータ点は緑色でグラフ上にプロットします。

In [ ]:
#@title 新規データの入力と確率計算（生データ版） { run: "auto" }

# 実測値の範囲に対応したスライダー設定
new_x1 = 6  #@param {type:"slider", min:4.0, max:8.0, step:0.1}
new_x2 = 2.5  #@param {type:"slider", min:1.5, max:4.5, step:0.1}


z = w1 * new_x1 + w2 * new_x2 + b
prob_class1 = 1.0 / (1.0 + np.exp(-z))
predicted_class = 1 if prob_class1 >= 0.5 else 0

print("-" * 60)
print(f"入力データ座標        : x1 = {new_x1:.1f} cm, x2 = {new_x2:.1f} cm")
print(f"パーセプトロン計算式  : z = ({w1:.1f}) * x1 + ({w2:.1f}) * x2 + ({b:.1f})")
print(f"計算値 (z)            : {z:.4f}")
print(f"Class 1 に属する確率  : P(y=1) = 1 / (1 + exp(-({z:.4f}))) = {prob_class1:.4f}")
print(f"分類結果              : {'Versicolor (Class 1)' if predicted_class == 1 else 'Setosa (Class 0)'}")
print("-" * 60)

x_min, x_max = 4.0, 8.0
y_min, y_max = 1.5, 4.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))

Z = np.where((w1 * xx + w2 * yy + b) >= 0, 1, 0)

plt.figure(figsize=(7, 6))
plt.contourf(xx, yy, Z, alpha=0.2, cmap="coolwarm")

if abs(w2) > 1e-5:
    x_vals = np.array([x_min, x_max])
    y_vals = -(w1 * x_vals + b) / w2
    plt.plot(x_vals, y_vals, color="black", linestyle="--", label="Decision Boundary")
elif abs(w1) > 1e-5:
    x_val = -b / w1
    plt.axvline(x=x_val, color="black", linestyle="--", label="Decision Boundary")

plt.scatter(X[y == 0, 0], X[y == 0, 1], color="blue", label="Setosa (Class 0)", s=60, edgecolors="k")
plt.scatter(X[y == 1, 0], X[y == 1, 1], color="red", label="Versicolor (Class 1)", s=60, edgecolors="k")
plt.scatter(new_x1, new_x2, color="green", label="New Data Point", s=140, edgecolors="k", marker="*")

plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.xlabel("x1: Sepal Length [cm]")
plt.ylabel("x2: Sepal Width [cm]")
plt.title(f"Classification of New Data Point\nProbability of Class 1: {prob_class1:.2%}")
plt.grid(True, linestyle=":")
plt.legend(loc="lower right")
plt.show()

## 課題

- 「重み」の値を調節してもらいました。これはネットワークの図においては＿＿＿＿に描かれる値であり、式においては ＿＿＿＿＿ にあたります。

- ご自身で設定した決定境界について、グラフ上部より書き写してください。

   ＿＿＿＿ $x_1 +$ ＿＿＿＿ $x_2 +$ ＿＿＿＿ $= 0$

- `x1 = 6.0`、`x2 = 2.5` に対する出力 $z$ は ＿＿＿＿ となりました。
これは、このアヤメが ＿＿＿＿＿＿＿＿＿＿＿ を表す値であり、

   $z =$ ＿＿＿＿ $\times 6.0 $ + ＿＿＿＿ $\times 2.5$ + ＿＿＿＿＿＿ で求められます。

- 「ニューラルネットワークを学習する」とは、＿＿＿＿＿＿＿＿＿＿＿＿ ことです。